# Example: BBBC021 — Mechanism of Action (MoA) Profiling

Perform morphological profiling on a multi-channel compound screen to analyze and predict the Mechanism of Action (MoA) of various treatments.

**Dataset:** [BBBC021](https://bbbc.broadinstitute.org/BBBC021) (3-channel fluorescent compound screen)

**Key Steps:**
1. **Metadata Integration:** Merge image metadata with compound MoA annotations.
2. **Multi-Channel Processing:** Link and process 3-channel images (Actin, Tubulin, DAPI).
3. **MoA Analysis:** Extract deep embeddings, compute extended properties, and classify compound effects within the phenospace.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Define the paths to the BBBC021 dataset and the metadata files, including the Mechanism of Action (MoA) annotations.

In [ ]:
DATA_DIR = "/datasets/BBBC/BBBC021/Week123"
METADATA_PATH = "/datasets/BBBC/BBBC021/metadata.csv"
MOA_PATH = "/datasets/BBBC/BBBC021/BBBC021_MoA_table_formated.csv"

## 2. Load & Merge Metadata

We load the image metadata and merge it with the MoA table to associate each treatment with its known biological mechanism.

In [ ]:
df_metadata = pd.read_csv(METADATA_PATH)
df_moa = pd.read_csv(MOA_PATH)

# Merge to associate images with MoA
df_merged = df_metadata.merge(
    df_moa,
    on=["Image_Metadata_Compound", "Image_Metadata_Concentration"],
    how="inner",
)

print(f"Merged metadata: {len(df_merged)} entries.")
df_merged.head()

## 3. Multi-Channel Data Discovery

BBBC021 consists of 3-channel images (Actin, Tubulin, DAPI). We specify the filename columns for each channel to ensure they are correctly linked and loaded.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    DATA_DIR,
    metadata_fn=make_dataframe_metadata_fn(
        df_merged,
        filename_column=["Image_FileName_Actin", "Image_FileName_Tubulin", "Image_FileName_DAPI"],
    ),
)
file_df.head()

## 4. Feature Extraction

We extract DINOv2 embeddings and compute an extended set of morphological properties to capture subtle phenotypic changes induced by the compounds.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=16,
    channel_mode="split",
    resize_size=1036,
    pad_size=1280,
)

df_properties = pheno.compute_properties(
    property_preset="full_extended",
    n_jobs=4,
)

## 5. Group Statistics

Aggregate morphological properties by compound and Mechanism of Action to identify characteristic phenotypic profiles.

In [ ]:
df_grouped = pheno.property_stats_by_group(
    group_by=["Image_Metadata_Compound", "Mechanism_of_action"],
    print_properties=["intensity_mean_ch0", "intensity_std_ch0"],
)

## 6. Visualization

Visualize the compound phenospace using t-SNE, colored by Mechanism of Action, to observe how different treatments cluster based on their morphological effects.

In [ ]:
pheno.plot_tsne(
    color_by="mechanism_of_action",
    n_components=2,
    exclude={"Image_metadata_compound": "taxol"},
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)

In [ ]:
pheno.plot_tsne(
    color_by="image_metadata_compound",
    n_components=2,
    filters={"mechanism_of_action": "Microtubule destabilizers"},
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)


## 7. Interactive Exploration

Launch the interactive explorer to visually inspect the 3-channel images and explore the relationship between compound concentration and phenotypic response.

In [ ]:
e = pheno.create_interactive_explorer(
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)